# Regularisation Lab: Linear and Logistic Regression

This lab uses the same datasets as the preceding lessons: Ames Housing for linear regression and the Wisconsin Diagnostic Breast Cancer dataset for logistic regression. We choose regularisation strength with cross-validation, then evaluate once on data held back from model selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml, load_breast_cancer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, log_loss, mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42

## Part 1: Regularised linear regression with Ames Housing

We predict sale price from above-ground living area. Polynomial features make the model flexible; Ridge regression adds an L2 penalty controlled by `alpha`, which corresponds to $\lambda$.

In [ ]:
ames = fetch_openml(data_id=42165, as_frame=True, parser='auto')
housing = ames.frame[['GrLivArea', 'SalePrice']].dropna().copy()
housing['GrLivArea_m2'] = housing['GrLivArea'].astype(float) * 0.092903
housing['SalePrice'] = housing['SalePrice'].astype(float)
housing = housing[(housing['GrLivArea_m2'] < 300) & (housing['SalePrice'] < 500_000)]

X_housing = housing[['GrLivArea_m2']]
y_housing = housing['SalePrice']
X_housing_dev, X_housing_test, y_housing_dev, y_housing_test = train_test_split(
    X_housing, y_housing, test_size=0.25, random_state=RANDOM_STATE
)
print(f'Development observations: {len(X_housing_dev)}')
print(f'Independent test observations: {len(X_housing_test)}')

In [ ]:
degree = 8
candidate_alphas = [0, 0.001, 0.01, 0.1, 1, 10, 100]
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
linear_results = []

for alpha in candidate_alphas:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        Ridge(alpha=alpha),
    )
    scores = cross_val_score(model, X_housing_dev, y_housing_dev, cv=cv, scoring='neg_root_mean_squared_error')
    linear_results.append({'alpha (lambda)': alpha, 'mean validation RMSE': -scores.mean()})

linear_results = pd.DataFrame(linear_results).sort_values('mean validation RMSE')
linear_results

In [ ]:
best_alpha = linear_results.iloc[0]['alpha (lambda)']
ridge_model = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    Ridge(alpha=best_alpha),
).fit(X_housing_dev, y_housing_dev)

test_predictions = ridge_model.predict(X_housing_test)
test_rmse = mean_squared_error(y_housing_test, test_predictions) ** 0.5
print(f'Selected alpha: {best_alpha}')
print(f'Independent test RMSE: ${test_rmse:,.0f}')

x_line = np.linspace(X_housing.min().iloc[0], X_housing.max().iloc[0], 300).reshape(-1, 1)
plt.figure(figsize=(9, 5))
plt.scatter(X_housing_dev, y_housing_dev, s=12, alpha=0.35, label='Development data')
plt.scatter(X_housing_test, y_housing_test, s=12, alpha=0.35, label='Test data')
plt.plot(x_line, ridge_model.predict(x_line), color='crimson', linewidth=3, label='Regularised polynomial')
plt.xlabel('Above-ground living area (m²)')
plt.ylabel('Sale price (USD)')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Part 2: Regularised logistic regression with Wisconsin Breast Cancer

`LogisticRegression` uses L2 regularisation by default. Its `C` parameter is the inverse of regularisation strength: a smaller `C` means stronger regularisation.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = cancer.target
X_cancer_dev, X_cancer_test, y_cancer_dev, y_cancer_test = train_test_split(
    X_cancer, y_cancer, test_size=0.20, stratify=y_cancer, random_state=RANDOM_STATE
)
print(f'Features: {X_cancer.shape[1]}')
print('Target labels:', dict(enumerate(cancer.target_names)))

candidate_c_values = [0.001, 0.01, 0.1, 1, 10, 100]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
logistic_results = []

for c_value in candidate_c_values:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=c_value, max_iter=5_000, random_state=RANDOM_STATE),
    )
    scores = cross_val_score(model, X_cancer_dev, y_cancer_dev, cv=cv, scoring='neg_log_loss')
    logistic_results.append({'C (inverse strength)': c_value, 'mean validation log loss': -scores.mean()})

logistic_results = pd.DataFrame(logistic_results).sort_values('mean validation log loss')
logistic_results

In [ ]:
best_c = logistic_results.iloc[0]['C (inverse strength)']
final_logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=best_c, max_iter=5_000, random_state=RANDOM_STATE),
).fit(X_cancer_dev, y_cancer_dev)

test_predictions = final_logistic_model.predict(X_cancer_test)
test_probabilities = final_logistic_model.predict_proba(X_cancer_test)[:, 1]
print(f'Selected C: {best_c}')
print(f'Independent test accuracy: {accuracy_score(y_cancer_test, test_predictions):.3f}')
print(f'Independent test log loss: {log_loss(y_cancer_test, test_probabilities):.3f}')

## Reflection

1. Which value of `alpha` produced the lowest validation RMSE for Ames Housing?
2. Why is test RMSE reported only after choosing `alpha`?
3. What happens to logistic-regression regularisation as `C` becomes smaller?
4. How is the purpose of regularisation the same in both tasks?